# XMCA-Net-Lite: 07 - Calibrated XMCA Combiner
### Purpose
Train the final LightGBM/XGBoost combiner on the training split, calibrate probabilities via Isotonic Regression on the validation split, and tune decision thresholds strictly on validation data.
**Strict Rule**: Never select thresholds or checkpoints using the test set!


In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path("/kaggle/working/xmca") if Path("/kaggle/working").exists() else Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, set_seed
from src.combiner.features import build_feature_dataframe
from src.combiner.model import XMCACombiner
from src.combiner.thresholds import tune_decision_threshold
from src.evaluation.metrics import compute_all_metrics
from src.utils.checkpoints import ShardManager
from src.utils.logging_metadata import save_run_metadata

config = load_config()
seed = config["project"]["default_seed"]
set_seed(seed)


In [ ]:
# Assemble unified feature tables for train, val, and test splits
p_df = ShardManager(Path(config["paths"]["perception_dir"])).load_all_shards("perception_scores")
r_df = ShardManager(Path(config["paths"]["retrieval_dir"])).load_all_shards("retrieval_scores")
u_df = ShardManager(Path(config["paths"]["utilization_dir"])).load_all_shards("utilization_scores")

unified = p_df[["id", "dataset", "s_p", "is_hallucination_gt"]].merge(
    r_df[["id", "s_r", "s_r_fixed", "retrieved_passages"]], on="id"
).merge(
    u_df[["id", "s_u", "s_u_text_only"]], on="id"
)
unified["evidence_count"] = unified["retrieved_passages"].apply(len)
unified["claim_count"] = 2

# Join with train/val/test split identities
splits_dir = Path(config["paths"]["splits_dir"])
train_ids = set()
val_ids = set()
test_ids = set()

for p in splits_dir.glob("*/*train.json"):
    with open(p) as f:
        train_ids.update([r["id"] for r in json.load(f)])
for p in splits_dir.glob("*/*val.json"):
    with open(p) as f:
        val_ids.update([r["id"] for r in json.load(f)])
for p in splits_dir.glob("*/*test.json"):
    with open(p) as f:
        test_ids.update([r["id"] for r in json.load(f)])

train_records = unified[unified["id"].isin(train_ids)].to_dict(orient="records")
val_records = unified[unified["id"].isin(val_ids)].to_dict(orient="records")
test_records = unified[unified["id"].isin(test_ids)].to_dict(orient="records")

print(f"Train: {len(train_records)}, Val: {len(val_records)}, Test: {len(test_records)}")


In [ ]:
# Build feature frames & train calibrated combiner
train_df = build_feature_dataframe(train_records)
val_df = build_feature_dataframe(val_records)
test_df = build_feature_dataframe(test_records)

combiner = XMCACombiner(model_type=config["models"]["combiner"]["model_type"], random_state=seed)
combiner.fit(train_df, val_df, target_col="target")

# Tune threshold strictly on validation set
val_probs = combiner.predict_proba(val_df)
best_thresh, val_metrics = tune_decision_threshold(val_df["target"].values, val_probs, metric="f1")
print(f"Optimal validation threshold: {best_thresh:.4f}")
print("Validation metrics at threshold:", val_metrics)

out_dir = Path(config["paths"]["combiner_dir"])
combiner.save(out_dir)

# Save threshold file permanently
with open(out_dir / "thresholds.json", "w", encoding="utf-8") as f:
    json.dump({"threshold": best_thresh, "val_metrics": val_metrics}, f, indent=2)

save_run_metadata(out_dir, config, seed, "07_combiner", {"best_threshold": best_thresh, "val_metrics": val_metrics})
print("\nStep 07 completed. Calibrated combiner model and thresholds saved.")
